In [23]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date
from pyspark.sql.functions import year
from pyspark.sql.functions import when
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("Lab 27 - EBAC").getOrCreate()

In [5]:
df_video = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/videos-stats.csv")
df_video = df_video.fillna(0, subset=["Likes", "Comments", "Views"])

In [6]:
df_comentario = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/comments.csv")

In [7]:
df_video.count()
df_comentario.count()

30036

In [8]:
df_video.dropna(subset=["Video ID"])
df_comentario.dropna(subset=["Video ID"])
df_video.count()
df_comentario.count()

30036

In [9]:
df_video.dropDuplicates(subset=["Video ID"])

DataFrame[_c0: int, Title: string, Video ID: string, Published At: date, Keyword: string, Likes: double, Comments: double, Views: double]

In [10]:
df_video = df_video.withColumn("Likes", col("Likes").cast(IntegerType())) \
  .withColumn("Comments", col("Comments").cast(IntegerType())) \
  .withColumn("Views", col("Views").cast(IntegerType()))

In [11]:
df_comentario = df_comentario.withColumn("Likes Comment", col("Likes").cast("int")) \
  .withColumn("Sentiment", col("Sentiment").cast("int")) \
  .drop("Likes")

In [12]:
df_video = df_video.withColumn("Interaction", col("Likes") + col("Comments") + col("Views"))

In [13]:
df_video = df_video.withColumn("Published At", to_date(col("Published At"), "dd-MM-yyyy"))

In [14]:
df_video = df_video.withColumn("Year", year(col("Published At")))

In [17]:
df_join_video_comments = df_video.join(df_comentario, df_video["Video ID"] == df_comentario["Video ID"], "inner").drop(df_comentario["Video ID"])

In [20]:
df_us_videos = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/USvideos.csv")

In [21]:
df_join_usvideos_video = df_video.join(df_us_videos, df_video["Title"] == df_us_videos["Title"], "inner").drop(df_us_videos["Title"])

In [24]:
null_count = df_video.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_video.columns])

In [25]:
df_video = df_video.drop("_c0")
df_video.write.mode("overwrite").parquet("videos-tratados-parquet")

In [26]:
df_join_video_comments = df_join_video_comments.drop("_c0")
df_join_video_comments.write.mode("overwrite").parquet("videos-comments-tratados-parquet")